In [1]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=30, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)

        # Build positional encoding matrix (max_len, d_model)
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()        # (30, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() *
            (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # even indices
        pe[:, 1::2] = torch.cos(position * div_term)   # odd indices
        pe = pe.unsqueeze(0)                            # (1, 30, d_model)

        # Register as buffer — not a parameter, but saved with model
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# v2
class StockTrendTransformer(nn.Module):
    def __init__(self, input_size=5, d_model=64, nhead=8, num_layers=3,
                 dim_feedforward=256, horizon=4, dropout=0.2,
                 max_len=30, pooling="last"):   # ← add pooling param
        super(StockTrendTransformer, self).__init__()
        self.d_model  = d_model
        self.horizon  = horizon
        self.pooling  = pooling               # ← store it

        self.input_proj = nn.Linear(input_size, d_model)

        # CLS token needed only if pooling="cls"
        if pooling == "cls":
            self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
            self.pos_encoder = PositionalEncoding(d_model, max_len + 1, dropout)
        else:
            self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = d_model,
            nhead           = nhead,
            dim_feedforward = dim_feedforward,
            dropout         = dropout,
            batch_first     = True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )

        self.bn      = nn.BatchNorm1d(d_model)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(d_model, horizon)

    def forward(self, x):
        out = self.input_proj(x)              # (batch, 30, d_model)

        # Prepend CLS token if needed
        if self.pooling == "cls":
            cls = self.cls_token.expand(x.size(0), -1, -1)  # (batch, 1, d_model)
            out = torch.cat([cls, out], dim=1)               # (batch, 31, d_model)

        out = self.pos_encoder(out)
        out = self.transformer_encoder(out)   # (batch, 30or31, d_model)

        # ── Pooling strategy ──────────────────────────────────────────────────
        if self.pooling == "last":
            out = out[:, -1, :]               # last timestep
        elif self.pooling == "mean":
            out = out.mean(dim=1)             # average all timesteps
        elif self.pooling == "cls":
            out = out[:, 0, :]               # CLS token at position 0

        out = self.bn(out)
        out = self.dropout(out)
        out = self.fc(out)
        return out

In [6]:
import torch
import torch.onnx

STOCKS = 'AMD'

# ---- 1. Load checkpoint ----
checkpoint = torch.load(
    f'Per Company Transformers/{STOCKS}/Transformer_best({STOCKS}).pth',
    map_location="cpu",
    weights_only=False
)

cfg        = checkpoint["config"]
state_dict = checkpoint["model_state_dict"]

print(f"val_loss from training: {checkpoint['val_loss']:.6f}")
print(f"Loaded config: {cfg}")

# ---- 2. Instantiate from saved config ----
model = StockTrendTransformer(
    input_size      = cfg["input_size"],
    d_model         = cfg["d_model"],
    nhead           = cfg["nhead"],
    num_layers      = cfg["num_layers"],
    dim_feedforward = cfg["dim_feedforward"],
    horizon         = cfg["horizon"],
    dropout         = 0.0,              # inference — dropout disabled
    max_len         = cfg["max_len"],
    pooling         = cfg["pooling"]
)

model.load_state_dict(state_dict)
model.eval()
print("Model loaded successfully ✓")

# ---- 3. Dummy input — (batch=1, seq_len=30, input_size=5) ----
dummy_input = torch.randn(1, cfg["window_size"], cfg["input_size"])

# ---- 4. Export to ONNX ----
torch.onnx.export(
    model,
    dummy_input,
    f'{STOCKS}_stock_trend_transformer.onnx',
    export_params       = True,
    opset_version       = 17,
    do_constant_folding = True,
    input_names         = ["input"],
    output_names        = ["output"],
    dynamic_axes        = {
        "input":  {0: "batch_size"},    # seq_len fixed at 30 (PE buffer constraint)
        "output": {0: "batch_size"}
    }
)


# ---- 5. Verify ----
import onnx
onnx_model = onnx.load(f'{STOCKS}_stock_trend_transformer.onnx')
onnx.checker.check_model(onnx_model)
print("ONNX model is valid ✓")

val_loss from training: 0.460786
Loaded config: {'input_size': 5, 'd_model': 64, 'nhead': 8, 'num_layers': 1, 'dim_feedforward': 128, 'horizon': 4, 'dropout': 0.1, 'window_size': 30, 'max_len': 30, 'pooling': 'mean', 'lr': 0.00021978434845411191, 'weight_decay': 9.413377095656487e-05, 'batch_size': 32, 'factor': 0.6000000000000001, 'scheduler_patience': 5}
Model loaded successfully ✓
ONNX model is valid ✓
